In [1]:

import pandas as pd
import numpy as np

df = pd.read_csv("project1_algorithmic_pricing_cleaned.csv")
df["date"] = pd.to_datetime(df["date"])

intervention_date = pd.Timestamp("2025-05-01")
regions = ["North", "South", "West"]

# Basic dataset checks
print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Regions:", sorted(df["region"].unique()))
print("Unique dates:", df["date"].nunique())

# Expected treatment assignment
expected_treatment = (
    (df["region"] == "North") &
    (df["date"] >= intervention_date)
).astype(int)

# Expected period assignment
expected_period = np.where(
    df["date"] < intervention_date,
    "pre_treatment",
    "post_treatment"
)

checks = {
    "Dataset has 540 rows and 11 columns": df.shape == (540, 11),
    "All three regions are present": set(df["region"].unique()) == set(regions),
    "180 unique dates": df["date"].nunique() == 180,
    "Intervention date exists": intervention_date in set(df["date"]),
    "Treatment assignment is correct": (df["treatment"] == expected_treatment).all(),
    "Period labels are correct": (df["period"] == expected_period).all(),
    "No missing values": df.isnull().sum().sum() == 0,
    "No exact duplicate rows": not df.duplicated().any(),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {name}")

print("\nPassed:", sum(checks.values()), "/", len(checks))


Dataset shape: (540, 11)
Date range: 2025-01-01 00:00:00 to 2025-06-29 00:00:00
Regions: ['North', 'South', 'West']
Unique dates: 180
PASS - Dataset has 540 rows and 11 columns
PASS - All three regions are present
PASS - 180 unique dates
PASS - Intervention date exists
PASS - Treatment assignment is correct
PASS - Period labels are correct
PASS - No missing values
PASS - No exact duplicate rows

Passed: 8 / 8


In [2]:

# Check treatment and period counts
print("PERIOD COUNTS")
print(df["period"].value_counts().sort_index())

print("\nREGION-WISE TREATMENT COUNTS")
print(pd.crosstab(df["region"], df["treatment"]))

print("\nREGION-WISE PERIOD COUNTS")
print(pd.crosstab(df["region"], df["period"]))

# Verify pre/post dates
pre = df[df["date"] < intervention_date]
post = df[df["date"] >= intervention_date]

print("\nPRE-TREATMENT")
print("Unique dates:", pre["date"].nunique())
print("Date range:", pre["date"].min(), "to", pre["date"].max())

print("\nPOST-TREATMENT")
print("Unique dates:", post["date"].nunique())
print("Date range:", post["date"].min(), "to", post["date"].max())

assert pre["date"].nunique() == 120
assert post["date"].nunique() == 60
assert (df.loc[df["region"] == "North", "treatment"].sum() == 60)
assert (df.loc[df["region"].isin(["South", "West"]), "treatment"].sum() == 0)

print("\nPASS - Period and donor-region checks")


PERIOD COUNTS
period
post_treatment    180
pre_treatment     360
Name: count, dtype: int64

REGION-WISE TREATMENT COUNTS
treatment    0   1
region            
North      120  60
South      180   0
West       180   0

REGION-WISE PERIOD COUNTS
period  post_treatment  pre_treatment
region                               
North               60            120
South               60            120
West                60            120

PRE-TREATMENT
Unique dates: 120
Date range: 2025-01-01 00:00:00 to 2025-04-30 00:00:00

POST-TREATMENT
Unique dates: 60
Date range: 2025-05-01 00:00:00 to 2025-06-29 00:00:00

PASS - Period and donor-region checks


In [3]:

# Daily revenue by region
daily_final = (
    df.pivot_table(
        index="date",
        columns="region",
        values="revenue",
        aggfunc="sum"
    )
    .reindex(columns=["North", "South", "West"])
    .reset_index()
)
daily_final.columns.name = None

# Weekly revenue by region (Monday-start weeks)
df_weekly = df.copy()
df_weekly["week_start"] = (
    df_weekly["date"]
    - pd.to_timedelta(df_weekly["date"].dt.dayofweek, unit="D")
)

weekly_final = (
    df_weekly.pivot_table(
        index="week_start",
        columns="region",
        values="revenue",
        aggfunc="sum"
    )
    .reindex(columns=["North", "South", "West"])
    .reset_index()
)
weekly_final.columns.name = None

# Save final CSVs
daily_final.to_csv("daily_wide_revenue.csv", index=False)
weekly_final.to_csv("weekly_wide_revenue.csv", index=False)

print("Daily output shape:", daily_final.shape)
print("Weekly output shape:", weekly_final.shape)
print("Daily missing values:", daily_final.isnull().sum().sum())
print("Weekly missing values:", weekly_final.isnull().sum().sum())
print("Daily duplicate dates:", daily_final["date"].duplicated().sum())
print("Weekly duplicate dates:", weekly_final["week_start"].duplicated().sum())

print("\nRevenue totals consistency:")
for region in ["North", "South", "West"]:
    daily_total = daily_final[region].sum()
    weekly_total = weekly_final[region].sum()
    print(
        region,
        "Daily:", round(daily_total, 2),
        "| Weekly:", round(weekly_total, 2),
        "| Match:", np.isclose(daily_total, weekly_total)
    )


Daily output shape: (180, 4)
Weekly output shape: (26, 4)
Daily missing values: 0
Weekly missing values: 0
Daily duplicate dates: 0
Weekly duplicate dates: 0

Revenue totals consistency:
North Daily: 11415313.37 | Weekly: 11415313.37 | Match: True
South Daily: 10220580.66 | Weekly: 10220580.66 | Match: True
West Daily: 12162190.59 | Weekly: 12162190.59 | Match: True
